In [6]:
import os
import pandas as pd

os.chdir("..") # Makes the working directory the root direcctory in order to import config
from src.config import PREPARED_DATA_DIR


col_subset_data = os.path.join(PREPARED_DATA_DIR, "data_columns_subset.dta")

df = pd.read_stata(
    col_subset_data,
    convert_categoricals=False,
)

col_subset_data_except_ID = [col for col in df.columns if col != 'DUPERSID']

print("Column datatypes:")
print(df.dtypes)


Column datatypes:
DUPERSID     object
TOTEXP23      int32
PHYEXE53       int8
OFTSMK53       int8
AGE23X         int8
SEX            int8
INSCOV23       int8
POVCAT23       int8
PERWT23F    float64
dtype: object


Need to look at the data types and what the atts represent to figure out what anomalies we can expect.

| Role | MEPS variable | Meaning |
|---|---|---|
| Outcome | `TOTEXP23` | Total healthcare expenditures in 2023 |
| Main predictor | `PHYEXE53` | Whether the person does at least 30 minutes of moderate or vigorous physical activity five times a week |
| Main predictor | `OFTSMK53` | How often the person smokes cigarettes |
| Comparison variable | `AGE23X` | Age at the end of 2023 |
| Comparison variable | `SEX` | Sex |
| Comparison variable | `INSCOV23` | Health insurance coverage in 2023 |
| Optional comparison variable | `POVCAT23` | Family income relative to the poverty line |
| Survey variable | `PERWT23F` | Final 2023 person weight |

In [9]:
vars_expected_to_be_binary = ['PHYEXE53', 'SEX']
vars_should_be_numeric = [col for col in col_subset_data_except_ID if col not in vars_expected_to_be_binary]
print(f"Variables expected to be binary: {vars_expected_to_be_binary}")
print(f"Variables should be numeric: {vars_should_be_numeric}")

Variables expected to be binary: ['PHYEXE53', 'SEX']
Variables should be numeric: ['TOTEXP23', 'OFTSMK53', 'AGE23X', 'INSCOV23', 'POVCAT23', 'PERWT23F']


All are numeric except the ID, so we can look at the range and see for negative vals.

In [ ]:
print('Numeric vars ranges: ')
descriptions = {
    "TOTEXP23": "Total healthcare expenditures in 2023",
    "PHYEXE53": "At least 30 minutes of moderate or vigorous physical activity five times a week",
    "OFTSMK53": "How often the person smokes cigarettes",
    "AGE23X": "Age at the end of 2023",
    "SEX": "Sex",
    "INSCOV23": "Health insurance coverage in 2023",
    "POVCAT23": "Family income relative to the poverty line",
    "PERWT23F": "Final 2023 person weight",
}

summary = df[vars_should_be_numeric].agg(["min", "max"]).T
summary.columns = ["Min", "Max"]
summary["Range"] = summary["Max"] - summary["Min"]
summary.index.name = "Variable"

summary.insert(0, "Description", summary.index.map(descriptions))

display(summary)

print('\nBinary vars value counts: ')
for col in vars_expected_to_be_binary:
    print( df[col].value_counts() )


Numeric vars ranges: 


,Min,Max,Range
Variable,,,
TOTEXP23,0.0,574675.000000,574675.000000
OFTSMK53,-8.0,3.000000,11.000000
AGE23X,-1.0,85.000000,86.000000
INSCOV23,1.0,3.000000,2.000000
POVCAT23,1.0,5.000000,4.000000
PERWT23F,0.0,131656.576994,131656.576994



Binary vars value counts: 
PHYEXE53
 1    7574
 2    7274
-1    3833
-8     128
-7     110
Name: count, dtype: int64
SEX
2    9906
1    9013
Name: count, dtype: int64
